# 03 - Hyperparameter Tuning: Neural Network (Optuna)

The Neural Network had the best ROC-AUC/PR-AUC/Precision at default settings but trailed Random Forest on Recall. Tuning architecture and training hyperparameters may close that gap, this is the key question this notebook answers.

**Practical note:** each trial trains a full small network for several epochs, this is meaningfully slower per trial than the tree models. Trial count is kept lower (20 instead of 50) to keep total runtime reasonable.


## Setup

In [1]:
import sys; print(sys.executable)

C:\Users\Ewis\Documents\Machine_learning_project\Heads-Up_IT3091\.venv\Scripts\python.exe


In [2]:
import time
t0 = time.time()
def step(msg): print(f"[{time.time()-t0:5.1f}s] {msg}", flush=True)

step("importing numpy/pandas/optuna")
import numpy as np
import pandas as pd
import optuna
step("importing torch (slow on first run)")
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import average_precision_score
step("imports done")

optuna.logging.set_verbosity(optuna.logging.WARNING)
torch.set_num_threads(4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ConfigurableMLP(nn.Module):
    def __init__(self, n_features, hidden1, hidden2, dropout1, dropout2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden1), nn.ReLU(), nn.Dropout(dropout1),
            nn.Linear(hidden1, hidden2), nn.ReLU(), nn.Dropout(dropout2),
            nn.Linear(hidden2, 1),
        )
    def forward(self, x):
        return self.net(x)

def objective(trial):
    hidden1 = trial.suggest_categorical('hidden1', [32, 64, 128])
    hidden2 = trial.suggest_categorical('hidden2', [16, 32, 64])
    dropout1 = trial.suggest_float('dropout1', 0.1, 0.5)
    dropout2 = trial.suggest_float('dropout2', 0.1, 0.5)
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [128, 256, 512])
    n_epochs = 15

    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train_np, dtype=torch.float32),
                      torch.tensor(y_train_np, dtype=torch.float32)),
        batch_size=batch_size, shuffle=True)

    model = ConfigurableMLP(n_features, hidden1, hidden2, dropout1, dropout2).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(n_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb).squeeze(1), yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(torch.tensor(X_val_np, dtype=torch.float32).to(device)).squeeze(1).cpu().numpy()
        y_score = 1 / (1 + np.exp(-val_logits))

    return average_precision_score(y_val, y_score)

step(f"defined everything, device = {device}")

[  0.0s] importing numpy/pandas/optuna
[  0.4s] importing torch (slow on first run)


C:\Users\Ewis\Documents\Machine_learning_project\Heads-Up_IT3091\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[  3.1s] imports done
[  3.2s] defined everything, device = cpu


## Load data

Same loading as the Random Forest notebook, plus feature scaling for the neural network.

In [3]:
from pathlib import Path
from sklearn.preprocessing import StandardScaler

train_df = pd.read_csv(Path('../../../data/processed/model_ready_train.csv'))
val_df   = pd.read_csv(Path('../../../data/processed/model_ready_val.csv'))

bool_cols = train_df.select_dtypes(include='bool').columns
train_df[bool_cols] = train_df[bool_cols].astype(int)
val_df[bool_cols]   = val_df[bool_cols].astype(int)

X_train = train_df.drop(columns=['Late_delivery_risk'])
y_train = train_df['Late_delivery_risk']
X_val   = val_df.drop(columns=['Late_delivery_risk'])
y_val   = val_df['Late_delivery_risk']

scaler = StandardScaler().fit(X_train)
X_train_np = scaler.transform(X_train).astype(np.float32)
X_val_np   = scaler.transform(X_val).astype(np.float32)
y_train_np = y_train.to_numpy().astype(np.float32)
y_val      = y_val.to_numpy()
n_features = X_train_np.shape[1]

print(X_train_np.shape, X_val_np.shape)   # expect (46026, 28) (7890, 28)
print("positive rate:", y_train_np.mean(), y_val.mean())

(46026, 28) (7890, 28)
positive rate: 0.5484509 0.5423320659062104


In [4]:
smoke = objective(optuna.trial.FixedTrial({
    'hidden1': 64, 'hidden2': 32, 'dropout1': 0.3, 'dropout2': 0.3,
    'lr': 1e-3, 'weight_decay': 1e-4, 'batch_size': 256}))
print("Smoke-test PR-AUC:", round(smoke, 4))

Smoke-test PR-AUC: 0.8295


## 1. Define a configurable network and the Optuna objective

In [11]:
import time

N_TRIALS = 30
start = time.time()

def progress_callback(study, trial):
    done = len(study.trials)
    elapsed = time.time() - start
    avg = elapsed / done
    remaining = avg * (N_TRIALS - done)
    print(
        f"Trial {done}/{N_TRIALS} | this trial: {trial.duration.total_seconds():.0f}s | "
        f"elapsed: {elapsed/60:.1f} min | ETA: {remaining/60:.1f} min | "
        f"best PR-AUC so far: {study.best_value:.4f}"
    )

study = optuna.create_study(direction='maximize', study_name='nn_prauc_tuning')
study.optimize(objective, n_trials=N_TRIALS, callbacks=[progress_callback])

print(f"Best validation PR-AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")


Trial 1/30 | this trial: 72s | elapsed: 1.2 min | ETA: 35.3 min | best PR-AUC so far: 0.8278
Trial 2/30 | this trial: 49s | elapsed: 2.0 min | ETA: 28.5 min | best PR-AUC so far: 0.8283
Trial 3/30 | this trial: 35s | elapsed: 2.6 min | ETA: 23.5 min | best PR-AUC so far: 0.8297
Trial 4/30 | this trial: 35s | elapsed: 3.2 min | ETA: 20.7 min | best PR-AUC so far: 0.8297
Trial 5/30 | this trial: 48s | elapsed: 4.0 min | ETA: 19.9 min | best PR-AUC so far: 0.8303
Trial 6/30 | this trial: 21s | elapsed: 4.3 min | ETA: 17.3 min | best PR-AUC so far: 0.8303
Trial 7/30 | this trial: 34s | elapsed: 4.9 min | ETA: 16.1 min | best PR-AUC so far: 0.8303
Trial 8/30 | this trial: 45s | elapsed: 5.6 min | ETA: 15.5 min | best PR-AUC so far: 0.8303
Trial 24/30 | this trial: 65s | elapsed: 15.9 min | ETA: 4.0 min | best PR-AUC so far: 0.8307
Trial 25/30 | this trial: 45s | elapsed: 16.7 min | ETA: 3.3 min | best PR-AUC so far: 0.8307
Trial 26/30 | this trial: 43s | elapsed: 17.4 min | ETA: 2.7 min | b

## 2. Run the study

**What we found:**

**No collapse this time, and the smoke test caught its own success before the full run even started** (PR-AUC 0.8295 on a fixed configuration, right in the healthy 0.82-0.84 range expected). That single check upfront is a good habit, it would have caught the earlier degenerate result in seconds instead of two wasted trials.

**Best validation PR-AUC after 30 trials: 0.8307, barely above the untuned network's 0.8301 (+0.0006).** This is a genuinely different result from Random Forest (+0.0032) and XGBoost (+0.0075), essentially no improvement from tuning. The best parameters (`lr=0.00133`, `hidden1=64`, `hidden2=32`, `batch_size=256`) all sit comfortably inside their search ranges, not pinned against a boundary the way earlier runs sometimes were, so this isn't a search-space limitation the way the first Random Forest attempt was. It looks like the untuned network's default architecture and settings were already close to as good as this particular search space allows.

**Runtime:** 30 trials took about 20 minutes, individual trials ranged from 21 to 72 seconds. Consistent with the earlier prediction that neural network tuning would be the slowest per-trial of the three models tuned.


**Note:** this cell originally asked to compare against the untuned network's Recall specifically, left over from before the objective was changed to PR-AUC. Recall is checked properly in Section 4 below instead, alongside every other metric, since Recall alone was exactly what caused the false "1.0 best score" result the first time this notebook was run. Kept here only as a record that the objective and the comparison approach both changed deliberately, not by accident.


## 3. Retrain the best configuration for longer and save

In [12]:
best_params = study.best_params
final_model = ConfigurableMLP(
    n_features, best_params['hidden1'], best_params['hidden2'],
    best_params['dropout1'], best_params['dropout2']
).to(device)

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train_np, dtype=torch.float32), torch.tensor(y_train_np, dtype=torch.float32)),
    batch_size=best_params['batch_size'], shuffle=True
)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params['lr'], weight_decay=best_params['weight_decay'])

N_EPOCHS_FINAL = 40  # longer run now that the configuration is fixed
for epoch in range(N_EPOCHS_FINAL):
    final_model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(final_model(xb).squeeze(1), yb)
        loss.backward()
        optimizer.step()

import joblib
torch.save(final_model.state_dict(), '../../../models/neural_network_tuned.pt')
joblib.dump(scaler, '../../../models/nn_scaler.pkl')

import json
with open('../../../data/processed/optuna_nn_best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)

print("Saved tuned Neural Network and best parameters.")


Saved tuned Neural Network and best parameters.


## 4. Full validation check: did tuning actually help?

Same two-part check used for XGBoost, since a single metric proved misleading there.


In [13]:
from sklearn.metrics import (recall_score, precision_score, f1_score, roc_auc_score,
                              average_precision_score, precision_recall_curve)

def evaluate_network(model):
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_val_np, dtype=torch.float32).to(device)).squeeze(1).cpu().numpy()
        score = 1 / (1 + np.exp(-logits))
        pred = (score >= 0.5).astype(int)
    return {
        'Recall': recall_score(y_val, pred), 'Precision': precision_score(y_val, pred),
        'F1': f1_score(y_val, pred), 'ROC-AUC': roc_auc_score(y_val, score),
        'PR-AUC': average_precision_score(y_val, score),
    }, score

tuned_metrics, tuned_score = evaluate_network(final_model)
untuned_metrics = {'Recall': 0.5857, 'Precision': 0.8458, 'F1': 0.6921, 'ROC-AUC': 0.7653, 'PR-AUC': 0.8301}

comparison = pd.DataFrame({'Untuned NN': untuned_metrics, 'Tuned NN': tuned_metrics}).round(4)
comparison['Change'] = (comparison['Tuned NN'] - comparison['Untuned NN']).round(4)
print("All metrics at the default 0.5 threshold:")
display(comparison)

def precision_at_recall(y_true, y_score, target_recall):
    prec, rec, _ = precision_recall_curve(y_true, y_score)
    valid = prec[rec >= target_recall]
    return valid.max() if len(valid) > 0 else float('nan')


rows = []
for target in [0.55, 0.60, 0.65, 0.70]:
    rows.append({'Target Recall': target, 'Tuned NN precision': precision_at_recall(y_val, tuned_score, target)})
pd.DataFrame(rows).round(4)


All metrics at the default 0.5 threshold:


,Untuned NN,Tuned NN,Change
Recall,0.5857,0.5866,0.0009
Precision,0.8458,0.8463,0.0005
F1,0.6921,0.6929,0.0008
ROC-AUC,0.7653,0.7647,-0.0006
PR-AUC,0.8301,0.8293,-0.0008


,Target Recall,Tuned NN precision
0,0.55,0.8620
1,0.60,0.8239
2,0.65,0.7513
3,0.70,0.7086


**What we found: essentially no change, in either direction, confirming Section 1's finding rather than contradicting it.**

| Metric | Untuned NN | Tuned NN | Change |
|---|---|---|---|
| Recall | 0.5857 | 0.5866 | +0.0009 |
| Precision | 0.8458 | 0.8463 | +0.0005 |
| F1 | 0.6921 | 0.6929 | +0.0008 |
| ROC-AUC | 0.7653 | 0.7647 | -0.0006 |
| PR-AUC | 0.8301 | 0.8293 | -0.0008 |

Every single difference is under 0.1 percentage point, smaller than the numbers vary between different random seeds. **ROC-AUC and PR-AUC did not move up together with Recall, but they did not fall off a cliff either, this is not the pattern from XGBoost's broken first attempt** (Precision -11pp, ROC-AUC -1.9pp), it's closer to noise around a flat line. The honest conclusion: tuning this network's hyperparameters (layer sizes, dropout, learning rate, weight decay, batch size) within this search space did not meaningfully change its behavior.

**This is a legitimate result worth reporting exactly as it is, not a failure to explain away.** Two candidate explanations, both reasonable: the fixed two-hidden-layer architecture may already be near its ceiling for this feature set and this amount of training data (46,026 rows, 28 features is not a lot by deep-learning standards), or the untuned defaults chosen in the model development stage happened to already be a solid configuration. Either way, the practical takeaway is the same: **this tuned network offers no real advantage over the untuned one**, so there is no reason to prefer the tuned version specifically, though the network overall still stands out for its strong ROC-AUC/PR-AUC/Precision from the original model comparison.

**For Stage 9's final candidate list:** with Random Forest and XGBoost both showing confirmed, genuine gains from tuning, and the Neural Network showing none, the practical case for carrying the network forward into threshold tuning and final test evaluation is weaker than it looked after the model development stage. Worth a group discussion on whether to keep it as a third option or focus the remaining stages on Random Forest and XGBoost specifically.
